In [38]:
from dotenv import load_dotenv
import os 
from langchain_groq import ChatGroq

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import CohereEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.agents import create_agent
from langchain_community.tools import tool
from langchain_cohere import CohereEmbeddings

In [39]:
load_dotenv()

True

In [40]:
groq_key=os.getenv("GROQ_API_KEY")
jina_key=os.getenv("JINA_API_KEY")
print("Environment Varibale loaded")


Environment Varibale loaded


loading our data

In [41]:
DATA_FILE_PATH=os.path.join("Data","hr_policy.txt")

Data Injestion


In [42]:
loader=TextLoader(DATA_FILE_PATH,encoding="utf-8")

documents=loader.load()

print("Data_Loaded")

print("="*40)



print(documents)

Data_Loaded
[Document(metadata={'source': 'Data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

Langchain Document

langchain process everything in form of documents

DOCUMENTS:

PAGE CONTENT: the actual data

MEATADATA: extra info about the data

In [43]:
len(documents)

1

In [44]:
print(f" total charcter  in the document:  {len(documents[0].page_content)}")

 total charcter  in the document:  2597


Spiltting our data


In [45]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks=text_splitter.split_documents(documents)

print(chunks)
print(len(chunks))

[Document(metadata={'source': 'Data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'Data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'Data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

Now each splitted chunk is a document -page_content and metadata

In [46]:
print(chunks[5])

page_content='5. REIMBURSEMENT POLICY
Employees can claim reimbursement for approved business expenses such as travel,
client meals, and internet bills used for official work.
All reimbursement claims must be submitted with valid bills within 30 days of the expense.
Claims are processed within 10 working days after approval from the reporting manager.' metadata={'source': 'Data\\hr_policy.txt'}


In [47]:
print(chunks[6])

page_content='6. CODE OF CONDUCT
Employees are expected to maintain professionalism and respect in the workplace.
Harassment, discrimination, or any form of workplace misconduct will not be tolerated
and may result in disciplinary action, including termination.
All employees must complete an annual Code of Conduct training.' metadata={'source': 'Data\\hr_policy.txt'}


EMBEDD OUR DATA

In [48]:


cohere_api_key = os.getenv("COHERE_API_KEY")

embedding_model = CohereEmbeddings(
    model="embed-english-light-v3.0",
    cohere_api_key=cohere_api_key
)

Store data in vector data base

In [49]:
vector_store=FAISS.from_documents(chunks,embedding_model)

print("CHUNKS ARE STORED IN VECTOR DB",vector_store.index.ntotal)

CHUNKS ARE STORED IN VECTOR DB 9


In [50]:
test_query="How many sick leave employes get"

top_matches=vector_store.similarity_search(test_query,k=2)
print(f"Query:{test_query}\n")
for i, match in enumerate(top_matches,start=1):
    print(match.page_content)
    print()

Query:How many sick leave employes get

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



In [51]:
#TOOL
retriever=vector_store.as_retriever(search_kwargs={"k":3})
@tool
def search_hr_policy(question:str)->str:
    """Search the HR policy document for information about leave,work from home,
     probation,notice period,reimbursement,code of conduct ,holiday,or exit process """
    matching_chunks=retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)


 Data Retrieval 
 
 LLM 

In [52]:
llm=ChatGroq(model="openai/gpt-oss-20b",temperature=0)

llm.model_name

'openai/gpt-oss-20b'

In [53]:
test_response=llm.invoke("Hey is laerning rag is hard answer in one line reply in ajoke")

In [54]:
test_response.content

'Learning rag is as hard as teaching a goldfish to do calculus—fun to watch, but you’ll still be the one who’s learning!'

AI AGENT

In [ ]:
hr_assistant=create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt="you are afriendly HR assistant working for acme crop." \
    "Always use the search_hr_policy tool to look up " \
    "fact before answering . if the answer isn't in the search result, say you dont know instead of guessing ."
    
)

print("HR assistantis ready to answer the quetion")

HR assistantis ready to answer the quetion


In [57]:
def ask_hr_assistant(question:str) -> str:
    """Send  a question to the rag agent and print a nicely formatted answer."""
    print("="*60)
    print("QUESTION:",question)
    print("-"*60)
    response=hr_assistant.invoke({"messages":[{"role":"user","content":question}]})
    answer=response["messages"][-1].content

    print("ANsWer",answer)
    print("="*60)

ask_hr_assistant("leave policy")

QUESTION: leave policy
------------------------------------------------------------
ANsWer Sure thing! Here’s a quick rundown of Acme Crop’s leave policy:

| Type of Leave | Entitlement | How to Request | Notes |
|---------------|-------------|----------------|-------|
| **Annual Paid Leave** | 20 days per calendar year | Submit through the HR portal at least **5 working days** in advance | Unused days can be carried forward up to **5 days** into the next year. |
| **Sick Leave** | 10 paid days per year | Same portal process | A medical certificate is required for any sick leave longer than **2 consecutive days**. |
| **Unpaid Leave (during Probation)** | Not applicable | Manager approval required | New hires on a 3‑month probation period are **not eligible for paid leave** but may take unpaid leave in emergencies. |

**Key points to remember**

- **Advance notice**: Always give at least 5 working days’ notice for annual or sick leave.
- **Carry‑over limit**: You can roll over a maximu